# 10.5 - Building MCP Servers

**Module 10 - Tool Use & MCP** | Netsetos GenAI Engineering

This notebook builds an MCP server from the ground up: a decorated function becomes a discoverable tool, three primitives (tools, resources, prompts) get sorted, and the same server object flips between a local stdio subprocess and a remote Streamable-HTTP endpoint. Each cell is a small, runnable stand-in that reproduces the SHAPE of FastMCP - the schema generation, the JSON-RPC handshake, the host config, the in-memory test, composition, and the Cloud Run deploy - so the protocol is tangible without needing a live host.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A one-line reproducibility marker. Several cells below call `round(...)` on fixed numbers, so there is nothing random to seed here - this comment just flags the intent that outputs are deterministic and should match the walkthroughs exactly.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A pure comment cell - no code executes. It exists as a header for the runnable stand-ins that follow, signalling that every printed number in this notebook is fixed and reproducible.

**How the code works - step by step**
- The single line is a comment, not a statement - running it does nothing.
- Its purpose is documentation: outputs in later cells are hardcoded arithmetic, so re-running always gives the same result.

**In one line:** a no-op marker that the notebook's outputs are deterministic.

**What you'll see:** (no output - environment prep)

## Setup - install FastMCP

Installs the two packages the lesson references: `fastmcp` (the standalone FastMCP 3.x library that powers most MCP servers) and `python-dotenv` (to load keys from a `.env` file). The runnable cells below use lightweight stand-ins, so this is mainly here so the illustrative `from fastmcp import ...` cells could run on a fresh Colab.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
!pip install python-dotenv fastmcp -q  # noqa


**Explanation**

The dependency-install cell. `!pip install` is a shell magic; `-q` quiets the log and `# noqa` stops linters flagging the magic.

**How the code works - step by step**
- **`python-dotenv`** - reads a local `.env` file into environment variables so keys aren't hardcoded.
- **`fastmcp`** - the FastMCP 3.x package: the `@mcp.tool()` / `@mcp.resource()` / `@mcp.prompt()` decorators and `mcp.run()`.

**In one line:** install FastMCP plus dotenv so the server and its client can run.

**What you'll see:** (no output - environment prep)

## Setup - environment keys

Seeds three provider API-key slots as empty environment variables. `setdefault` only sets a value if the key is missing, so a real key already exported in your shell (or loaded from `.env`) is left untouched. Building an MCP server needs no key at all - these are here for the multi-provider demos elsewhere in the module.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Standard key-loading boilerplate: read provider keys from the environment rather than hardcoding them. Nothing in this notebook actually calls a model, so blank values are fine.

**How the code works - step by step**
- **`import os`** - access to the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** and the Anthropic / Google lines - create the slot only if it isn't already set, so an existing key wins.
- The trailing comments point to where each key is issued (platform.openai.com, console.anthropic.com, aistudio.google.com).

**In one line:** register key slots without overwriting real keys, never hardcoding secrets.

**What you'll see:** (no output - environment prep)

## 1 - The three primitives: tools DO, resources READ, prompts TEMPLATE

MCP has exactly three primitives, and choosing the right one matters. This cell builds `MiniMCP`, a ~10-line stand-in for FastMCP, so you can see the SHAPE of what a decorator does: it registers your function into one of three buckets. Tools are actions the model invokes, resources are addressable read-only data, prompts are reusable templates.

In [ ]:
# THE THREE PRIMITIVES - a tiny stand-in for FastMCP to show the SHAPE (tools DO, resources READ, prompts TEMPLATE).
class MiniMCP:
    def __init__(self, name): self.name=name; self.tools={}; self.resources={}; self.prompts={}
    def tool(self):
        def deco(fn): self.tools[fn.__name__] = fn; return fn
        return deco
    def resource(self, uri):
        def deco(fn): self.resources[uri] = fn; return fn
        return deco
    def prompt(self):
        def deco(fn): self.prompts[fn.__name__] = fn; return fn
        return deco

mcp = MiniMCP("Netsetos")

@mcp.tool()
def add_gst(amount):                                # a TOOL: an action the model invokes
    return round(amount * 1.18, 2)
@mcp.resource("catalog://courses")
def courses():                                      # a RESOURCE: read-only data the client GETs
    return [{"id": "genai-bootcamp", "price_inr": 14999}]
@mcp.prompt()
def quote(course):                                  # a PROMPT: a reusable template
    return f"Price of {course} with GST, one line."

print("  Server 'Netsetos' exposes:")
print(f"    tools     = {list(mcp.tools)}       (DO: invoke an action)")
print(f"    resources = {list(mcp.resources)}   (READ: GET addressable data)")
print(f"    prompts   = {list(mcp.prompts)}          (TEMPLATE: reusable message)")
print(f"  client calls a tool: add_gst(14999) -> {mcp.tools['add_gst'](14999)}")

# Output:
#   Server 'Netsetos' exposes:
#     tools     = ['add_gst']       (DO: invoke an action)
#     resources = ['catalog://courses']   (READ: GET addressable data)
#     prompts   = ['quote']          (TEMPLATE: reusable message)
#   client calls a tool: add_gst(14999) -> 17698.82

**Explanation**

A tiny teaching mock, not the real library. `MiniMCP` shows that `@mcp.tool()`, `@mcp.resource()`, and `@mcp.prompt()` are just decorators that stash a function into a dict keyed by name (or URI). One server object exposes any mix of the three.

**How the code works - step by step**
- **`MiniMCP.__init__`** - creates three empty dicts: `tools`, `resources`, `prompts`.
- **`tool()` / `resource(uri)` / `prompt()`** - each returns a decorator that files the function into its bucket (`resource` keys by the URI you pass).
- **`add_gst` / `courses` / `quote`** - one function per primitive: `add_gst` DOES (returns `amount * 1.18`), `courses` READS (returns catalog data), `quote` is a TEMPLATE string.
- The prints list each bucket, then call `mcp.tools['add_gst'](14999)` directly - exactly what a real client does over JSON-RPC in cell 7.

**In one line:** three decorators sort your functions into DO / READ / TEMPLATE, and a client lists then calls them.

**What you'll see:** Prints the server's three buckets - `tools = ['add_gst']`, `resources = ['catalog://courses']`, `prompts = ['quote']` - and a live tool call: `add_gst(14999) -> 17698.82`.

## 2 - Effectful tools and hint annotations

Most real tools DO something with side effects - write a row, send a message, move money. MCP lets you tag that intent with **tool hints** (`readOnlyHint`, `destructiveHint`, `idempotentHint`) so the host can shape the UX: auto-run a safe read, but ask the user to confirm a destructive action. This cell uses the real `FastMCP` API to show where those annotations go.

> **Uses `fastmcp`** (installed above) - no API key or model call needed; nothing is executed, it defines two decorated functions for reference.

In [ ]:
# EFFECTFUL TOOLS - most real tools DO something (write a row, move money). Annotate the intent.
from fastmcp import FastMCP
mcp = FastMCP("Netsetos")

@mcp.tool(annotations={"readOnlyHint": True})       # safe: only reads, no side effects
def get_invoice(invoice_id: str) -> dict:
    "Look up an invoice."
    return {"id": invoice_id, "status": "paid"}

@mcp.tool(annotations={"destructiveHint": True, "idempotentHint": False})  # mutates; unsafe to retry
def refund_invoice(invoice_id: str) -> dict:
    "Refund an invoice - a real side effect (moves money)."
    # ... perform the refund ...
    return {"id": invoice_id, "refunded": True}

# Output: (illustrative) annotations are advisory HINTS the host reads to shape the UX -
#         it may auto-run a readOnlyHint tool but ASK the user to confirm a destructiveHint one.
#         Hints are not enforcement: still validate inputs and guard effectful tools yourself.

**Explanation**

An illustrative definition cell - it registers two tools but calls nothing, so it produces no runtime output. The point is the `annotations={...}` argument on `@mcp.tool()`: advisory metadata the host reads to decide whether to auto-run or confirm.

**How the code works - step by step**
- **`get_invoice`** - annotated `readOnlyHint: True`; it only reads, so a host may auto-run it.
- **`refund_invoice`** - annotated `destructiveHint: True, idempotentHint: False`; it mutates state (moves money) and is unsafe to retry, so a host should ask the user to confirm.
- The hints are **advisory, not enforcement** - you still validate inputs and guard effectful tools yourself (cell 10).

**In one line:** annotate a tool's intent so the host can gate risky actions, but never rely on hints for safety.

**What you'll see:** No printed output - the cell only defines two annotated tools. The trailing comment explains that a host reads the hints to auto-run reads and confirm destructive calls.

## 3 - Transports: stdio vs Streamable HTTP

The same server code runs local or remote - only the *transport* changes. **stdio** (the default) means the client launches your server as a subprocess and talks over stdin/stdout; that is how desktop hosts run a LOCAL server. **Streamable HTTP** is a network endpoint remote clients connect to, and it replaced the older HTTP+SSE transport. This cell models the switch as a single argument.

In [ ]:
# TRANSPORTS - the SAME server runs two ways; you change ONE argument, not the code.
import os
def run(transport="stdio", host="0.0.0.0", port=None):
    if transport == "stdio":
        return "stdio: client launches this file as a SUBPROCESS; talk over stdin/stdout (local hosts)"
    if transport == "streamable-http":
        port = port or int(os.environ.get("PORT", 8000))
        return f"streamable-http: serving on http://{host}:{port} (remote clients; replaced SSE)"
    raise ValueError("unknown transport")

print("  Local dev  -> mcp.run()")
print("    ", run("stdio"))
print("  Production -> mcp.run(transport='streamable-http', host='0.0.0.0', port=int(os.environ['PORT']))")
print("    ", run("streamable-http", port=8080))

# Output:
#   Local dev  -> mcp.run()
#      stdio: client launches this file as a SUBPROCESS; talk over stdin/stdout (local hosts)
#   Production -> mcp.run(transport='streamable-http', host='0.0.0.0', port=int(os.environ['PORT']))
#      streamable-http: serving on http://0.0.0.0:8080 (remote clients; replaced SSE)

**Explanation**

A stand-in `run()` that just names what each transport means, so the difference is concrete. The lesson's point: you flip `mcp.run(transport=...)`, not rewrite the functions or schemas.

**How the code works - step by step**
- **`run(transport="stdio")`** - returns the local story: the client spawns this file as a subprocess and pipes JSON-RPC over stdin/stdout.
- **`run(transport="streamable-http")`** - reads `PORT` from the env (default 8000) and returns a real endpoint on `host:port` for remote clients; notes it replaced SSE.
- An unknown transport raises `ValueError`.
- The prints contrast `mcp.run()` (local dev) with `mcp.run(transport='streamable-http', host='0.0.0.0', port=int(os.environ['PORT']))` (production).

**In one line:** same server, one argument switches between a local subprocess and a remote HTTP endpoint.

**What you'll see:** Two labelled lines: `Local dev -> mcp.run()` describing the stdio subprocess, and `Production -> ... streamable-http` serving on `http://0.0.0.0:8080` for remote clients.

## 4 - The client and the JSON-RPC 2.0 handshake

Everything a client does is JSON-RPC 2.0, and the lifecycle is always the same: **initialize** (version + capabilities handshake), then **list** (discover the tools/resources/prompts), then **call**. A host like Claude does this for you; when you write your own client, FastMCP's async `Client` is three lines. This cell builds the actual wire messages so the protocol is tangible.

In [ ]:
# THE CLIENT + JSON-RPC 2.0 HANDSHAKE - every MCP client: initialize -> discover -> call.
def rpc(method, params=None, _id=[0]):
    _id[0] += 1
    return {"jsonrpc": "2.0", "id": _id[0], "method": method, "params": params or {}}

handshake = [
    rpc("initialize", {"protocolVersion": "2025-11-25", "capabilities": {}}),
    rpc("tools/list"),
    rpc("tools/call", {"name": "add_gst", "arguments": {"amount": 14999}}),
]
labels = ["handshake (version + capabilities)", "discover the tools", "invoke a tool"]
print("  The three JSON-RPC messages every MCP client sends:")
for msg, lab in zip(handshake, labels):
    print(f"    {msg['method']:12s} (id={msg['id']}) - {lab}")
resp = {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"type": "text", "text": "17698.82"}]}}
print(f"  server replies -> result.content[0].text = {resp['result']['content'][0]['text']}")

# With FastMCP the real client is 3 lines (async):
#   from fastmcp import Client
#   async with Client("server.py") as c:      # or Client(mcp) in-memory
#       await c.call_tool("add_gst", {"amount": 14999})

# Output:
#   The three JSON-RPC messages every MCP client sends:
#     initialize   (id=1) - handshake (version + capabilities)
#     tools/list   (id=2) - discover the tools
#     tools/call   (id=3) - invoke a tool
#   server replies -> result.content[0].text = 17698.82

**Explanation**

A message-builder demo, not a live connection. `rpc()` assembles JSON-RPC envelopes and the cell prints the exact three-message sequence any MCP client sends, plus a mocked server reply, so you can see the shape that flows over any transport.

**How the code works - step by step**
- **`rpc(method, params)`** - builds a JSON-RPC 2.0 message (`jsonrpc`, auto-incrementing `id`, `method`, `params`); the mutable-default `_id` is what advances the id.
- **`handshake`** - the three-call lifecycle: `initialize` (protocol version + capabilities), `tools/list` (discover), `tools/call` (invoke `add_gst` with `amount: 14999`).
- The loop prints each method with its id and a plain-English label.
- **`resp`** - a mocked server reply; the answer nests under `result.content[0].text`, showing MCP results are typed content blocks.
- The commented block shows the real 3-line FastMCP `Client` that does all of this for you.

**In one line:** initialize -> list -> call, all JSON-RPC 2.0, with results returned as typed content blocks.

**What you'll see:** Prints the three client messages (`initialize` id=1, `tools/list` id=2, `tools/call` id=3) with labels, then the server's reply `result.content[0].text = 17698.82`.

## 5 - Point a desktop host at your server

To use your server in Claude Desktop (or Cursor / VS Code), you tell the host how to launch it via `claude_desktop_config.json` - an `mcpServers` object mapping a name to the `command` + `args` (and optional `env`) the host runs to spawn your stdio server. This cell builds and validates that config, and shows the one-command shortcut.

In [ ]:
# HOST CONFIG - point Claude Desktop (Cursor / VS Code) at your server via claude_desktop_config.json.
import json
def desktop_config(name, command, args, env=None):
    entry = {"command": command, "args": args}
    if env: entry["env"] = env
    return {"mcpServers": {name: entry}}

cfg = desktop_config("netsetos", "python", ["-m", "netsetos_server"],
                     env={"NETSETOS_API_KEY": "..."})
print("  claude_desktop_config.json (the host spawns your stdio server):")
print("   ", json.dumps(cfg, separators=(",", ":")))
srv = cfg["mcpServers"]["netsetos"]
ok = bool(srv.get("command")) and isinstance(srv.get("args"), list)
print(f"  valid launch spec? {ok}  (command + args -> host can spawn it; restart the host to load)")
print("  Or run: fastmcp install claude-desktop server.py   (writes this config for you)")

# Output:
#   claude_desktop_config.json (the host spawns your stdio server):
#     {"mcpServers":{"netsetos":{"command":"python","args":["-m","netsetos_server"],"env":{"NETSETOS_API_KEY":"..."}}}}
#   valid launch spec? True  (command + args -> host can spawn it; restart the host to load)
#   Or run: fastmcp install claude-desktop server.py   (writes this config for you)

**Explanation**

A config-builder plus a sanity check. `desktop_config` produces the exact JSON a host reads at startup; the validation confirms the launch spec has the two things a host needs to spawn the subprocess.

**How the code works - step by step**
- **`desktop_config(name, command, args, env)`** - assembles `{"mcpServers": {name: {command, args, env?}}}`, adding `env` only when provided.
- **`cfg`** - a `netsetos` server launched via `python -m netsetos_server`, with a secret passed through `env` (never hardcoded in the server).
- **`ok`** - checks the entry has a truthy `command` and a list `args`; a missing one means the host can never start the server.
- The final line shows `fastmcp install claude-desktop server.py`, which writes this config for you; Cursor and VS Code use the same shape.

**In one line:** a host spawns your local server from `command` + `args` in `claude_desktop_config.json` - restart the host to load it.

**What you'll see:** Prints the compact `claude_desktop_config.json` JSON, `valid launch spec? True`, and the `fastmcp install claude-desktop` shortcut line.

## 6 - Test the tool and the generated schema

You don't test an MCP server by clicking around in Claude. Because `@mcp.tool()` generates the input schema from your function signature, a bad type hint is a bad schema - so testing the tool also tests the schema. This cell reimplements what the decorator does (reflect the signature into JSON Schema) and runs an in-memory assertion, the same idea as FastMCP's in-process test `Client`.

In [ ]:
# TEST + DEBUG - @mcp.tool() builds the schema from your signature; test with an in-memory client.
import inspect, json
JSON = {int: "integer", float: "number", str: "string", bool: "boolean"}
def schema_from(fn):
    props, required = {}, []
    for pname, p in inspect.signature(fn).parameters.items():
        props[pname] = {"type": JSON.get(p.annotation, "string")}
        if p.default is inspect._empty: required.append(pname)
    return {"type": "object", "properties": props, "required": required,
            "description": (fn.__doc__ or "").strip()}

def add_gst(amount: float) -> float:
    "Add 18% GST to a rupee amount."
    return round(amount * 1.18, 2)

print("  @mcp.tool() generates this inputSchema from add_gst(amount: float):")
print("   ", json.dumps(schema_from(add_gst), separators=(",", ":")))
got = add_gst(100)                                  # an in-memory client call: no subprocess, no network
assert got == 118.0, got
print(f"  in-memory test: call_tool('add_gst', amount=100) -> {got}  (assert == 118.0 PASSED)")

# Output:
#   @mcp.tool() generates this inputSchema from add_gst(amount: float):
#     {"type":"object","properties":{"amount":{"type":"number"}},"required":["amount"],"description":"Add 18% GST to a rupee amount."}
#   in-memory test: call_tool('add_gst', amount=100) -> 118.0  (assert == 118.0 PASSED)

**Explanation**

A measurement harness, not a model call. `schema_from` mirrors the decorator's schema generation via `inspect`, and the cell then calls the tool directly and asserts the result - exactly how you'd unit-test in CI with no subprocess or network.

**How the code works - step by step**
- **`JSON`** - maps Python types (`int`, `float`, `str`, `bool`) to JSON Schema type names.
- **`schema_from(fn)`** - walks the signature: each parameter becomes a typed property, and parameters with no default are marked `required`; the docstring becomes the schema description.
- **`add_gst(amount: float)`** - the tool under test, with a hint and a docstring that together ARE the schema.
- The prints show the generated `inputSchema`, then `add_gst(100)` is called in-process and asserted `== 118.0` - a passing pytest-style check.

**In one line:** the signature generates the schema, so an in-memory call asserts both the logic and the schema at once.

**What you'll see:** Prints the generated `inputSchema` (`amount` -> `number`, required, with the docstring as description) and `in-memory test: call_tool('add_gst', amount=100) -> 118.0 (assert == 118.0 PASSED)`.

## 7 - Fail cleanly with ToolError

What happens when a tool fails? Validate inputs and `raise ToolError(...)` so the client gets a clean, intentional error (a `tools/call` result flagged `isError`) instead of a server traceback - and the model can read the message and retry. This cell shows the real FastMCP pattern.

> **Uses `fastmcp`** - definition only, nothing executes and no key is needed.

In [ ]:
# WHEN A TOOL RAISES - validate inputs and raise ToolError; the client gets a clean error, not a crash.
from fastmcp import FastMCP
from fastmcp.exceptions import ToolError

mcp = FastMCP("Netsetos")

@mcp.tool()
def add_gst(amount: float) -> float:
    "Add 18% GST to a rupee amount (amount must be non-negative)."
    if amount < 0:
        raise ToolError("amount must be >= 0")   # a controlled, client-visible error
    return round(amount * 1.18, 2)

# Output: (illustrative) add_gst(-5) does NOT crash the server. FastMCP catches the ToolError and
#         returns a tools/call RESULT with isError=true carrying your message:
#         {"content":[{"type":"text","text":"amount must be >= 0"}],"isError":true}
#         The model SEES that message and can retry with a valid argument. A raised ToolError is
#         reported to the client; an unhandled exception is caught too, but its detail is masked.

**Explanation**

An illustrative definition - it registers a guarded tool but calls nothing, so there is no runtime output. The point is the `raise ToolError(...)` on invalid input, which FastMCP turns into a controlled, client-visible error rather than a crash.

**How the code works - step by step**
- **`from fastmcp.exceptions import ToolError`** - the exception FastMCP surfaces to the client as a clean error.
- **`add_gst`** - validates that `amount >= 0`; on a negative value it raises `ToolError("amount must be >= 0")` instead of returning garbage or crashing.
- FastMCP catches the raised `ToolError` and returns a `tools/call` result with `isError: true` carrying your message; the model sees it and can retry.
- An unhandled exception is caught too, but its detail is masked - an explicit `ToolError` is the message you control.

**In one line:** validate inputs and raise `ToolError` so failures come back as recoverable, client-visible errors.

**What you'll see:** No printed output - the cell only defines the guarded tool. The trailing comment shows the error result shape (`isError: true` with your message) that a client would receive for `add_gst(-5)`.

## 8 - Compose and scale servers

Real deployments combine servers. FastMCP 3.0 lets you **mount** a local sub-server, **proxy** a remote one, or turn a whole OpenAPI REST API into MCP tools - without rewriting anything. On a name clash, prefixing keeps components apart and the most-recently-mounted server wins. This cell models the merge-and-prefix behaviour with plain dicts.

In [ ]:
# COMPOSE - FastMCP 3.0 merges servers with mount(prefix=...); on a name clash, the last mount wins.
def mount(base, sub, prefix):
    for name, fn in sub.items():
        base[f"{prefix}_{name}"] = fn                # prefixing avoids clashes
    return base

weather = {"forecast": lambda: "34C"}
billing = {"forecast": lambda: "invoice #42"}        # clashes with weather on 'forecast'
combined = {}
mount(combined, weather, "weather")
mount(combined, billing, "billing")
print(f"  Two sub-servers mounted -> one tool surface: {sorted(combined)}")

proxy = dict(combined)
proxy["remote_search"] = lambda: "(proxied from a remote MCP server)"   # FastMCP.as_proxy()
print(f"  FastMCP.as_proxy() adds a REMOTE server's tools as if local -> {len(proxy)} tools total")

# The real FastMCP calls:
#   main.mount(billing_server, prefix="billing")     # compose a LOCAL sub-server
#   proxy = FastMCP.as_proxy("https://remote/mcp")   # expose a REMOTE server as if local
#   FastMCP.from_openapi(spec, client)               # turn a whole REST API into MCP tools

# Output:
#   Two sub-servers mounted -> one tool surface: ['billing_forecast', 'weather_forecast']
#   FastMCP.as_proxy() adds a REMOTE server's tools as if local -> 3 tools total

**Explanation**

A stand-in for FastMCP's composition. `mount` prefixes each sub-server's tool names so two servers that both define `forecast` coexist; the cell also mimics `as_proxy` by folding in a remote server's tool as if it were local.

**How the code works - step by step**
- **`mount(base, sub, prefix)`** - copies each sub-server function into `base` under `prefix_name`, avoiding collisions.
- **`weather`** and **`billing`** - two servers that both define `forecast`; mounting with prefixes yields `weather_forecast` and `billing_forecast`.
- **`proxy`** - adds `remote_search`, standing in for `FastMCP.as_proxy(url)` exposing a remote server's tools on the same surface.
- The commented block shows the real calls: `main.mount(sub, prefix=...)`, `FastMCP.as_proxy(url)`, `FastMCP.from_openapi(spec, client)`.

**In one line:** prefix-namespace mounted servers, proxy remote ones, and present one merged tool surface to the client.

**What you'll see:** Prints the combined tool surface `['billing_forecast', 'weather_forecast']` and `FastMCP.as_proxy() ... -> 3 tools total` after folding in the proxied remote tool.

## 9 - Deploy: stdio to Cloud Run

Shipping is the same server on the remote transport, containerized. Locally you run `mcp.run()` (stdio); for production you run `mcp.run(transport="streamable-http", host="0.0.0.0", port=int(os.environ["PORT"]))` inside a Docker image on Cloud Run, which scales across client connections. This cell maps each environment to its transport and shows the minimal Dockerfile.

In [ ]:
# DEPLOY - local stdio for dev; Streamable HTTP in a container on Cloud Run for production.
def deploy_target(where):
    if where == "local":
        return ("stdio", "client spawns a subprocess; no network")
    if where == "cloud-run":
        return ("streamable-http", "container binds 0.0.0.0:$PORT; Cloud Run fans out across clients")
    raise ValueError("unknown target")

for where in ["local", "cloud-run"]:
    transport, note = deploy_target(where)
    print(f"  {where:9s} -> transport={transport:15s} | {note}")

# Dockerfile (essentials):
#   FROM python:3.12-slim
#   RUN pip install fastmcp
#   COPY server.py .
#   CMD ["python", "server.py"]   # server.py: mcp.run(transport="streamable-http",
#                                 #             host="0.0.0.0", port=int(os.environ["PORT"]))

# Output:
#   local     -> transport=stdio           | client spawns a subprocess; no network
#   cloud-run -> transport=streamable-http | container binds 0.0.0.0:$PORT; Cloud Run fans out across clients

**Explanation**

A tiny mapping demo tying the whole notebook together: local dev is stdio, production is a Streamable-HTTP container. It reinforces that deploy is a re-run with a different transport, not a rewrite.

**How the code works - step by step**
- **`deploy_target(where)`** - returns the transport plus a one-line note for `local` (stdio subprocess, no network) and `cloud-run` (streamable-http binding `0.0.0.0:$PORT`, fanned out across clients).
- The loop prints both rows side by side.
- The commented **Dockerfile** is the essentials: slim Python base, `pip install fastmcp`, copy `server.py`, and run it - with the server binding `0.0.0.0` and reading the injected `$PORT`, never a hardcoded port.
- Note the security warning in the surrounding lesson: a Streamable-HTTP server is public by default, so put a `TokenVerifier` in front before exposing real tools.

**In one line:** same server, remote transport, in a container - local stdio for dev, Streamable-HTTP on Cloud Run for production.

**What you'll see:** Prints two rows: `local -> transport=stdio` (subprocess, no network) and `cloud-run -> transport=streamable-http` (binds `0.0.0.0:$PORT`, Cloud Run fans out across clients).

Across these cells you built the whole arc of an MCP server without a single real host running: a function plus a decorator IS the server and the schema is generated from your type hints; you pick the right primitive - tools DO, resources READ, prompts TEMPLATE; the same server rides stdio locally or Streamable HTTP remotely by one argument; any client reaches it through initialize -> list -> call; and you wire it into a desktop host, fail cleanly with ToolError, and compose servers with mount/proxy. The through-line is "write the function once, every AI client can use it." Next, Lesson 10.6 takes the Streamable-HTTP server from the deploy cell and does the full remote Cloud Run walkthrough; Lesson 10.7 adds production hardening (auth depth, rate limits, observability).